## Импорты

In [1]:
import sys
import os
from pathlib import Path

# Допустим, что ноутбук находится в той же директории, что и папка acoustic/
sys.path.insert(0, str(Path.cwd()))

import warnings
warnings.filterwarnings("ignore")
import warnings
warnings.filterwarnings("ignore", message="Non-default generation parameters")

In [2]:
from acoustic.utils.config import load_config
from acoustic.dataset.load_dataset import load_and_prepare_dataset
from acoustic.models.load_model import build_model
from acoustic.training.load_metrics import load_metrics
from acoustic.training.callbacks import get_callback
from acoustic.training import get_trainer_class


## Загрузка конфигурации

In [3]:
CONFIG_PATH = "acoustic/configs/base_config.yaml"

cfg = load_config(CONFIG_PATH, overrides=None)

print("Configuration loaded")

Configuration loaded


## Загрузка датасета

In [4]:
dataset = load_and_prepare_dataset(cfg)

Applying duration filter: 100%|██████████| 2/2 [00:00<00:00, 516.60it/s]


train: kept 2561 / 2562 examples
validation: kept 355 / 356 examples


Applying filters: 100%|██████████| 2/2 [00:00<00:00, 157.52it/s]


Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/20 [00:00<?, ? examples/s]

## Инициализация модели

In [5]:
model, processor, data_collator = build_model(cfg)

print("Model built")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Model built


## Создание и загрузка метрик, callbacks, trainer

In [6]:
metrics_list = load_metrics(cfg['training']['metrics'])
print(f"Metrics: {cfg['training']['metrics']}")

callbacks = []
for cb_name in cfg['training']['callbacks']:
    callbacks.append(get_callback(cb_name))

Metrics: ['wer', 'cer']


In [7]:

trainer_name = cfg['training'].get('trainer', 'BaseTrainer')
TrainerClass = get_trainer_class(trainer_name)

trainer = TrainerClass(
    cfg=cfg,
    model=model,
    processor=processor,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    metrics=metrics_list,
    callbacks=callbacks,
    data_collator=data_collator
)

## Обучение

In [8]:
print("Starting training")
trainer.train()

Starting training


Training:  20%|██        | 3/15 [00:30<02:01, 10.11s/step]

{'loss': 1.8855, 'learning_rate': 6.000000000000001e-08, 'epoch': 0.96}
{'eval_loss': 1.7496228218078613, 'eval_wer': 0.2776, 'eval_cer': 0.0555, 'eval_runtime': 8.1898, 'eval_samples_per_second': 2.442, 'eval_steps_per_second': 0.366, 'epoch': 0.96}


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}


KeyboardInterrupt: 

## Проверка

In [ ]:
eval_dataset = dataset['validation']

print("Demo on validation examples")
import random
import torch

if eval_dataset and len(eval_dataset) > 0:
    indices = random.sample(range(len(eval_dataset)), min(3, len(eval_dataset)))
    for i in indices:
        example = eval_dataset[i]
        audio_array = example["audio"]["array"]
        
        inputs = processor.feature_extractor(
            audio_array, 
            sampling_rate=16000, 
            return_tensors="pt"
        )
        input_features = inputs.input_features.to(next(model.parameters()).device)
        
        with torch.no_grad():
            predicted_ids = model.generate(input_features)
        
        pred_text = processor.decode(predicted_ids[0], skip_special_tokens=True)
        ref_text = example["sentence"]
        
        print(f"\nExample {i+1}:")
        print(f" Reference: {ref_text}")
        print(f" Prediction: {pred_text}")
else:
    print("No validation dataset for demo.")

Demo on validation examples


`use_cache = True` is incompatible with gradient checkpointing. Setting `use_cache = False`...



Example 4:
 Reference: тысячи лет назад человек по имени аристарх сказал что солнечная система вращается вокруг солнца
 Prediction:  Тысячи лет назад человек по имени Рестарх сказал, что солнечная система вращается вокруг солнца.

Example 1:
 Reference: они умеют отлично видеть в темноте при помощи ночного видения и почти незаметно передвигаться оцелоты выслеживают добычу сливаясь с окружающей обстановкой а затем набрасываются на добычу
 Prediction:  Они умеют отлично видеть стимноте при помощи ночного видения и почти незаметно передвигаться. Отсилоты выслеживают добычу, сливаясь с окружающей обстановкой, а затем набрасываются на добычу.

Example 9:
 Reference: вариант становящийся всё более популярным для тех кто планирует взять академический год это путешествовать и учиться
 Prediction:  Вариант, становящийся всё более популярным для тех, кто планирует взять академический год, — это путешествовать и учиться.
